# Improve a RAG agent from user feedback

You will build a customer-support agent for Brewline, an online shop for coffee gear. The
agent has two tools. It searches the shop's policies, and it looks up orders. Then you will
make it better, using only the answers your users marked as wrong.

| Piece | What it does | Who runs it |
|---|---|---|
| `search_policies` | searches the shop's policies, stored as vectors in Milvus | **your code** |
| `lookup_order` | reads one order from the shop's order data | **your code** |
| `brewline-support` | the agent's prompt, stored in AcruxCore so it can change without a deploy | **the platform** |
| the optimizer | reads the feedback and drafts rewrites of the prompt | **the platform** |
| `compare_local` | runs **your** agent on the live prompt and on each rewrite, side by side | **your code** |

This is the order of work:

1. store the prompt, build the agent, and run it on six questions,
2. mark the two wrong answers with a thumbs-down and a comment,
3. turn that feedback into a dataset, and let the optimizer draft three rewrites,
4. run the **whole agent** on each rewrite, tools and all, and grade the answers,
5. promote the rewrite that wins,
6. break it on purpose four times, so you know what each mistake looks like.

Step 4 is the reason this notebook exists. The optimizer scores a rewrite with one model
call per example, and that call has no tools. For an agent, that score says whether the
*wording* is better. It cannot say whether the agent now searches before it answers. Only a
run of your own code can show that.

Every cell runs against a real account. Nothing here is faked or mocked. Model answers change
from run to run, so your scores and rewrites will not match the saved output exactly.

**Two ways to do every step.** Each step that creates something has two headings:
**In the dashboard**, with the values to enter, and **The same thing in code**, with a cell to
run. They are not two different features. The dashboard and the SDK call the same API, so the
result is the same. Pick either one. Doing both is harmless, because every code cell looks for
what already exists before it creates anything.

**Three kinds of code cell.** Most of this notebook is not the agent. Every code cell's
lead-in says which kind it is:

| Label | What it is | Goes in your app? |
|---|---|---|
| **Setup** | creates something on the platform, once. The dashboard does the same job. | no |
| **Your app** | the code that would really ship | **yes** |
| **Check** | proves the step worked, or shows what just happened | no |

The failure cells near the end are marked **broken on purpose**. The last section lists
exactly which cells add up to the real app.

**Companion tutorial:** [Improve a RAG agent from user feedback](https://docs.acruxcore.com/docs/tutorials/improve-a-rag-agent-from-feedback).
The same steps as standalone scripts are in
[`scripts/tutorials/improve-a-rag-agent-from-feedback/python/`](https://github.com/AcruxCore/AcruxCore/tree/main/scripts/tutorials/improve-a-rag-agent-from-feedback/python).

---

## Step 0 — What you need before you start

**1. An account and an API key.** Sign up at [acruxcore.com](https://acruxcore.com). Then go
to **Account & keys → New key** and copy the key. It is shown once.

**2. A connected model provider, and two models.** A new account cannot reach any model yet.
Under **Gateway → Credentials**, add a provider key. Then under **Gateway → Models**, add two
models that use it:

- `gpt-4o-mini` runs the agent.
- `gpt-4.1` drafts the rewrites and grades the answers. Drafting and grading are harder jobs
  than one answer, so they get the stronger model.

The names are the **public names** you type on the Models page. To use other models, change
`MODEL` and `STRONG_MODEL` in the cell below.
Step by step with screenshots:
[Route your app's LLM calls through the gateway](https://docs.acruxcore.com/docs/guides/route-calls-through-the-gateway).

**3. A running Milvus.** The agent searches the shop's policies in
[Milvus](https://milvus.io), an open-source vector database. The
[Milvus deployment overview](https://milvus.io/docs/install-overview.md) lists the ways to get
one. Set `MILVUS_URI` to its address; the default is `http://localhost:19530`.

**4. Install the packages.** `pymilvus[model]` is the Milvus client plus a small embedding
model that runs on your machine, so you need no extra API key for the vectors.
`compare_local`, `commit_candidate` and `pin_prompts` are recent in `acruxcore`, so upgrade
rather than install.

In [ ]:
%pip install -q --upgrade acruxcore "pymilvus[model]"

Now set your key and the base URL.

**Setup.** Real environment variables are better. The two `os.environ` lines are here so the
notebook runs on its own. A key typed into a notebook is saved inside the notebook file, so
do not commit it.

In [1]:
import json
import os

# Better: set these in your shell before you start Jupyter.
os.environ.setdefault("ACRUXCORE_API_KEY", "acx_sk_...")
os.environ.setdefault("ACRUXCORE_BASE_URL", "https://api.acruxcore.com/api/v1")

MODEL = "gpt-4o-mini"        # runs the agent - a public name from Gateway -> Models
STRONG_MODEL = "gpt-4.1"     # drafts the rewrites and grades the answers
PROMPT = "brewline-support"  # the prompt this notebook creates

#: The date the agent treats as today. Fixed, so the orders' dates give the same answers on
#: every run. A real app passes the real date.
TODAY = "2026-10-26"

# Where your Milvus runs.
MILVUS_URI = os.environ.get("MILVUS_URI", "http://localhost:19530")

### Preflight

**Check.** Run this before anything else. It checks four things, in the order they usually
fail, so you get one clear message instead of a stack trace from a later cell.

In [2]:
import httpx

import acruxcore
from acruxcore import AcruxCore

hub = AcruxCore()  # reads ACRUXCORE_API_KEY and ACRUXCORE_BASE_URL

# 1. Does the key work at all?
await hub.prompts.list(limit=1)
print("api key: ok")

# 2. Is this SDK new enough? Steps 8 and 9 need these three calls.
missing = [
    name
    for name, ok in [
        ("runs.commit_candidate", hasattr(hub.runs, "commit_candidate")),
        ("experiments.compare_local", hasattr(hub.experiments, "compare_local")),
        ("pin_prompts", hasattr(acruxcore, "pin_prompts")),
    ]
    if not ok
]
print("sdk features: ok" if not missing else f"!! acruxcore {acruxcore.__version__} lacks {missing} - upgrade it")

# 3. Are both models on this team? There is no SDK method for the model list, so this
#    one call goes to the REST API directly.
async with httpx.AsyncClient() as http:
    res = await http.get(
        f"{os.environ['ACRUXCORE_BASE_URL']}/gateway/models",
        headers={"Authorization": f"Bearer {os.environ['ACRUXCORE_API_KEY']}"},
    )
models = [m["publicName"] for m in res.json()]
for name in (MODEL, STRONG_MODEL):
    print(f"model {name!r}:", "ok" if name in models else "MISSING - add it under Gateway -> Models")

# 4. Can we reach Milvus?
from pymilvus import MilvusClient

milvus = MilvusClient(MILVUS_URI)
milvus.list_collections()
print("milvus: ok")

api key: ok
sdk features: ok
model 'gpt-4o-mini': ok
model 'gpt-4.1': ok
milvus: ok


---

## Step 1 — Two different tests for one rewrite

Read this step before any code. The rest of the notebook depends on it.

### The general problem

An agent's answer depends on two things. One is the prompt's wording. The other is what the
agent *does*: which tools it calls, what it searches for, and what comes back. A change to the
prompt can change both. A test that sees only the wording cannot tell you about the second.

### Where AcruxCore sits

The prompt lives in AcruxCore as **versions**. A version never changes after you commit it.
An **alias**, such as `production`, is a name that points at one version. Your app asks for
`production`, so moving the alias changes what users get, with no deploy.

The **optimizer** reads the feedback, drafts rewrites, and scores each one. To score, it
renders the rewrite with each example's variables and makes **one model call, with no tools**.
For a prompt that only writes text, that is the whole job. For this agent, the model cannot
search or look anything up in that call, so the score compares wording only.

**`compare_local`** is the second test. It runs **your own agent function** once per variant,
at the same time, over the same examples:

| Variant | What `prompts.render("brewline-support", "production")` returns inside it |
|---|---|
| `production` | the version the alias points at, as usual |
| `candidate-A` | the version that candidate was committed as |

Your agent code does not change. The SDK holds the pinned version in a context variable, so
two variants can share one client and run at the same time without seeing each other's pin.
The alias still decides the tools, so only the template changes.

### What this means for you

Use the optimizer to *draft* rewrites. Use `compare_local` to *choose* one, whenever your
prompt drives tools or retrieval. A rewrite that wins the optimizer's table can still lose
when the tools are real, and the other way round.

---

## Step 2 — Store the prompt

The system message is short on purpose. It says to use the tools, but not *when*. That gap is
what the feedback will fix later. `{{ today }}` and `{{ question }}` are template
variables that your code fills in when it renders the prompt.

The first version you commit also creates two aliases, `production` and `staging`, both
pointing at version 1.

### In the dashboard

**Prompts → New prompt**, then add the two messages in the editor and **Commit new version**.

| Field | What to enter |
|---|---|
| **Name** | `brewline-support` |
| **Description** | Brewline customer support agent. |
| **Message 1** | role **system** — paste the `SYSTEM` text from the next cell, `{{ today }}` included |
| **Message 2** | role **user** — `{{ question }}` |
| **Default model** | `gpt-4o-mini`, or the public name of your own model |

### The same thing in code

**Setup.** It looks for the prompt first, and commits version 1 only when the prompt has no
version yet. A second run, or a run after you clicked through the dashboard, creates nothing.

In [3]:
SYSTEM = (
    "You are the customer support assistant for Brewline, an online shop for coffee "
    "grinders, espresso machines and brewing gear. Today is {{ today }}.\n\n"
    "Be friendly and answer in two to four sentences. Use the tools when you need "
    "store information."
)

found = [p for p in (await hub.prompts.list(search=PROMPT)).data if p.name == PROMPT]
if found:
    prompt = found[0]
    print(f"prompt {PROMPT} already exists")
else:
    prompt = await hub.prompts.create(PROMPT, description="Brewline customer support agent.")
    print(f"created prompt {PROMPT}")

if (await hub.prompts.list_versions(prompt.id)).total == 0:
    version = await hub.prompts.commit_version(
        prompt.id,
        messages=[
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": "{{ question }}"},
        ],
        model=MODEL,
    )
    print(f"committed v{version.version_number} on {MODEL}")
else:
    print("the prompt already has a version - nothing committed")

live = await hub.prompts.render(PROMPT, "production", {"question": "-", "today": TODAY})
print(f"production serves v{live.version_number}")

created prompt brewline-support
committed v1 on gpt-4o-mini
production serves v1


---

## Step 3 — Build the agent

### The shop's data

**Setup.** In a real app this data lives in your own systems: the policies in a document
store, the orders in a database. Here it is inline, so the notebook runs on its own. This is
the same text as the tutorial's `data/` folder.

In [4]:
POLICIES = {
    "grinder-care": """# Grinder care

Clean the burrs of a grinder every two weeks with grinder cleaning tablets. Never wash the burrs with water.

If a grinder makes a loud clicking noise, a stone or a bean husk may be stuck between the burrs. Unplug it, remove the upper burr and brush the chamber clean.

If a grinder stops grinding finely, the burrs may be worn. Burrs usually last about 500 kilograms of coffee.""",
    "returns": """# Returns and refunds

Unused items can be returned within 30 days of delivery for a full refund to the original payment method.

Grinders and espresso machines that have been used can be returned within 14 days of delivery. A used grinder or machine is refunded as store credit, not money, because we have to clean and recalibrate it before it can be sold again.

After the return window closes, an item can no longer be returned. Grinders and machines are still covered by the warranty, which pays for repairs.

To start a return, the customer replies to their order confirmation email. We send a prepaid label within one business day.""",
    "shipping": """# Shipping

Orders ship from our warehouse in Rotterdam within one business day.

Standard shipping takes 3 to 5 business days inside the EU and is free on orders over 60 euros. Below that it costs 6 euros.

Express shipping takes 1 to 2 business days inside the EU and costs 15 euros.

Orders to the United Kingdom take 5 to 8 business days. The customer pays UK import VAT on delivery.

When an order ships, the customer gets an email with a tracking link.""",
    "warranty": """# Warranty

Grinders and espresso machines have a 2-year warranty from the delivery date. It covers motors, burrs that wear out early, pumps and heating elements. We repair the item for free, including shipping both ways. If we cannot repair it, we replace it.

Kettles, scales and other accessories have a 1-year warranty.

The warranty does not cover damage from dropping the item, descaling it with vinegar, or grinding anything other than coffee beans.

To make a warranty claim, the customer sends their order number and a short video of the problem to support.""",
}

ORDERS = {
    "BR-1042": {"customer": "Ana", "delivered_on": "2026-09-14", "status": "delivered",
                "items": [{"name": "Fellow Ode Gen 2 grinder", "category": "grinder", "used": True, "price_eur": 345}]},
    "BR-1077": {"customer": "Malik", "delivered_on": "2026-10-20", "status": "delivered",
                "items": [{"name": "Niche Zero grinder", "category": "grinder", "used": True, "price_eur": 649}]},
    "BR-1088": {"customer": "Sofia", "delivered_on": "2026-10-02", "status": "delivered",
                "items": [{"name": "Hario Buono kettle", "category": "accessory", "used": False, "price_eur": 49}]},
    "BR-1101": {"customer": "Tom", "delivered_on": None, "status": "shipped", "shipped_on": "2026-10-23",
                "destination": "United Kingdom",
                "items": [{"name": "Acaia Pearl scale", "category": "accessory", "used": False, "price_eur": 159}]},
    "BR-0954": {"customer": "Lena", "delivered_on": "2025-06-30", "status": "delivered",
                "items": [{"name": "Gaggia Classic Pro espresso machine", "category": "espresso machine", "used": True, "price_eur": 449}]},
}

print(f"{len(POLICIES)} policy documents, {len(ORDERS)} orders")

4 policy documents, 5 orders


### Index the policies in Milvus

**Setup.** This cell turns each policy paragraph into a vector and stores it in a Milvus
collection called `brewline_policies`. A **vector** is a list of numbers that stands for the
meaning of a text. Two texts about the same thing get vectors that are close together, so a
search finds a paragraph by meaning, not only by shared words.

In a real app, indexing is not part of the agent. A separate job runs it when a policy
changes. The agent only searches. That is why this cell is **Setup** and the next is
**Your app**.

The embedding model is downloaded once, the first time it runs (about 100 MB), so this cell
can take a minute. If the collection already holds every paragraph, the cell indexes nothing.

In [5]:
from pymilvus import model

COLLECTION = "brewline_policies"

# A small sentence-embedding model that runs on this machine, on the CPU.
embed = model.DefaultEmbeddingFunction()

# One chunk per paragraph. Headings carry no policy, so they are skipped.
chunks = [
    {"source": source, "text": para.strip()}
    for source, document in sorted(POLICIES.items())
    for para in document.split("\n\n")
    if para.strip() and not para.strip().startswith("#")
]


def stored_rows() -> int:
    """How many paragraphs the collection holds. A notebook helper, not part of pymilvus."""
    if not milvus.has_collection(COLLECTION):
        return 0
    return milvus.query(COLLECTION, filter="id >= 0", output_fields=["count(*)"])[0]["count(*)"]


if stored_rows() == len(chunks):
    print(f"collection {COLLECTION} already holds all {len(chunks)} paragraphs")
else:
    if milvus.has_collection(COLLECTION):
        milvus.drop_collection(COLLECTION)  # out of date: rebuild it from POLICIES
    milvus.create_collection(COLLECTION, dimension=embed.dim)
    vectors = embed.encode_documents([c["text"] for c in chunks])
    milvus.insert(COLLECTION, [{"id": i, "vector": v, **c} for i, (c, v) in enumerate(zip(chunks, vectors))])
    print(f"indexed {len(chunks)} paragraphs into {COLLECTION} ({embed.dim} numbers per vector)")

collection brewline_policies already holds all 16 paragraphs


### The search and the two tools

**Your app.** This is the agent's retrieval and its tools. `search_policy_store` turns the
question into a vector with the same model, and asks Milvus for the closest paragraphs.

The `@acrux.tool` decorator turns each function's signature and docstring into the tool
definition the model reads. So the docstring is not a comment for your team. The model reads
it on every call, and it decides when the tool is used.

In [6]:
from typing import Any, Dict, List, Tuple

from acruxcore import acrux


def search_policy_store(query: str, k: int = 4) -> List[Dict[str, str]]:
    """The k policy paragraphs closest in meaning to the query, closest first."""
    hits = milvus.search(COLLECTION, embed.encode_queries([query]), limit=k, output_fields=["source", "text"])[0]
    return [{"source": hit["entity"]["source"], "text": hit["entity"]["text"]} for hit in hits]


@acrux.tool
async def search_policies(query: str) -> List[Dict[str, str]]:
    """Search Brewline's store policies (returns, refunds, warranty, shipping, grinder care).

    Args:
        query: What to look for, in a few words, e.g. "return used grinder".
    """
    return search_policy_store(query)


@acrux.tool
async def lookup_order(order_id: str) -> Dict[str, Any]:
    """Look up one customer order: its items, whether they were used, and its delivery date.

    Args:
        order_id: The order number, e.g. "BR-1042".
    """
    order = ORDERS.get(order_id.strip().upper())
    return {"order_id": order_id, **order} if order else {"error": f"No order {order_id}."}

**Check.** Search Milvus directly, with no model involved, so you know the right paragraph is
there to be found.

In [7]:
for hit in search_policy_store("grinder loud clicking noise", k=2):
    print(f"[{hit['source']}] {hit['text'][:90]}...")

[grinder-care] If a grinder makes a loud clicking noise, a stone or a bean husk may be stuck between the ...
[warranty] Grinders and espresso machines have a 2-year warranty from the delivery date. It covers mo...


### The agent

**Your app.** One function. It renders the `production` alias, then lets the SDK run the tool
loop: the model asks for a tool, the SDK runs your function, and the result goes back to the
model until it answers. It takes the client as an argument, the same as the tutorial's
`agent.py`.

The `trace` argument is optional. In step 9 a test run passes its own trace in, so the agent's
spans for one test example land in that example's trace.

In [8]:
from typing import Optional


async def ask(hub: AcruxCore, question: str, trace: Optional[Dict[str, Any]] = None) -> Tuple[str, Optional[str]]:
    """Answer one customer question. Returns the answer and the trace id."""
    rendered = await hub.prompts.render(PROMPT, "production", {"question": question, "today": TODAY})
    result = await hub.gateway.run_prompt_with_tools(
        rendered,
        tools=[search_policies, lookup_order],
        trace={"name": PROMPT, **(trace or {})},
    )
    return result.content, result.trace_id

---

## Step 4 — Run it and read the traces

**Your app.** Six questions a real customer might send. Each answer prints its trace id.

In [9]:
QUESTIONS = [
    "Hi, I used my grinder from order BR-1042 a few times and I don't like it. Can I send it back for a refund?",
    "My Niche Zero from order BR-1077 arrived last week and I've used it twice. Can I return it for my money back?",
    "Can I return the kettle from order BR-1088? I never opened the box.",
    "Where is my order BR-1101? It should have arrived by now.",
    "The pump on my espresso machine from order BR-0954 stopped working. What can I do?",
    "My grinder is making a loud clicking noise. Is it broken?",
]

trace_ids: Dict[str, str] = {}
for question in QUESTIONS:
    answer, trace_id = await ask(hub, question)
    trace_ids[question] = trace_id
    print(f"Q: {question}\nA: {answer}\n")

Q: Hi, I used my grinder from order BR-1042 a few times and I don't like it. Can I send it back for a refund?
A: Since your order was delivered on September 14, 2026, unfortunately, you are now outside the 14-day return window. Used grinders can only be returned for store credit within that period. However, your grinder is covered by a 2-year warranty, which means we can repair it for free if you encounter any issues. If you have any specific concerns, please let me know!

Q: My Niche Zero from order BR-1077 arrived last week and I've used it twice. Can I return it for my money back?
A: Since you have used the Niche Zero grinder from order BR-1077, it can be returned within 14 days of delivery, but you'll only receive store credit, not a refund in money. Since it arrived on October 20, you still have time to initiate the return. If you have any questions about the process, feel free to ask!

Q: Can I return the kettle from order BR-1088? I never opened the box.
A: Yes, you can return t

Read the last two answers against the policies in step 3.

- **BR-1101.** The shipping policy says UK orders take 5 to 8 business days, and the customer
  pays import VAT on delivery. An order shipped on 23 October is not late yet.
- **The clicking grinder.** The grinder care policy says a loud click usually means a stone or
  a bean husk is stuck between the burrs.

On most runs the agent uses neither policy. The trace shows why:

![The BR-1101 trace: an LLM span, a lookup_order tool span and a second LLM span, with no search_policies span](https://docs.acruxcore.com/img/tutorials/improve-a-rag-agent-from-feedback/trace-tool-spans.png)

**Check.** Read the same thing from the API instead of a screenshot: which tools each answer
called. The SDK reports tool spans in the background, so flush first.

In [10]:
def tool_calls(spans) -> List[str]:
    """Names of the tool spans in a span tree, in order. A notebook helper, not SDK."""
    names: List[str] = []
    for span in spans:
        if span.kind == "tool":
            names.append(span.name)
        names += tool_calls(span.children)
    return names


await hub.gateway.flush()
for question, trace_id in trace_ids.items():
    detail = await hub.traces.get(trace_id)
    print(f"{', '.join(tool_calls(detail.spans)) or 'no tool':<34} {question[:60]}")

lookup_order, search_policies      Hi, I used my grinder from order BR-1042 a few times and I d
lookup_order, search_policies      My Niche Zero from order BR-1077 arrived last week and I've 
lookup_order, search_policies      Can I return the kettle from order BR-1088? I never opened t
lookup_order                       Where is my order BR-1101? It should have arrived by now.
lookup_order, search_policies      The pump on my espresso machine from order BR-0954 stopped w
no tool                            My grinder is making a loud clicking noise. Is it broken?


Look at the last rows. The BR-1101 answer looked up the order but never searched the shipping
policy. The grinder answer usually calls no tool at all. The system message says "use the tools
when you need store information", and for these two questions the model decides it does not
need them. The answers sound helpful, so nothing errors. They are just not Brewline's policy.

Model output varies, so your list may differ in a row. The questions about returns and the
warranty usually call both tools, because the words "return" and "warranty" point the model at
a policy. The two questions above do not.

---

## Step 5 — Mark the wrong answers

Feedback is what the optimizer learns from. So write each comment as the **correct answer**,
not only "wrong". A comment that says what good looks like can later grade an answer. A bare
"wrong" cannot.

### In the dashboard

Open each trace under **Observability → Traces**. Under **Feedback**, click the thumbs-down
**▼**, fill in the two fields and save.

| Field | What to enter |
|---|---|
| **Rating** | thumbs-down **▼** |
| **Label** | `wrong-answer` |
| **Comment** | the matching text from `FEEDBACK` in the next cell — one trace per entry |

![The grinder trace with the thumbs-down feedback and its comment saved under Feedback](https://docs.acruxcore.com/img/tutorials/improve-a-rag-agent-from-feedback/feedback-on-trace.png)

### The same thing in code

**Setup.** In a real team a person clicks the thumbs-down, or a support tool sends it over the
API. This cell does the same with `hub.traces.submit_feedback`. It first reads the
thumbs-down already left on this prompt's traces, so a second run adds no duplicate.

In [11]:
GRINDER = QUESTIONS[5]
SHIPPING = QUESTIONS[3]

FEEDBACK = {
    GRINDER: (
        "Wrong. Our grinder care guide says a loud click usually means a stone or bean husk is "
        "stuck between the burrs. Correct answer: it is probably not broken; unplug the grinder, "
        "remove the upper burr and brush the chamber clean."
    ),
    SHIPPING: (
        "Too vague. Orders to the UK take 5 to 8 business days, so an order shipped on 23 October "
        "is not late yet. Correct answer: expect it by 4 November, and UK import VAT is paid on "
        "delivery. This is in our shipping policy."
    ),
}


async def thumbs_down_on(prompt_id: str):
    """Every thumbs-down left on a trace that ran this prompt, read page by page.

    A notebook helper, NOT an SDK function. The one real call inside it is
    hub.traces.list_feedback(), filtered on the server by rating and prompt.
    """
    rows, page = [], 1
    while True:
        result = await hub.traces.list_feedback(page=page, limit=100, rating="down", prompt_id=prompt_id)
        rows += result.data
        if page * result.limit >= result.total:
            return rows
        page += 1


already = {row.comment for row in await thumbs_down_on(prompt.id)}
for question, comment in FEEDBACK.items():
    if comment in already:
        print(f"feedback already left: {question[:50]}")
        continue
    feedback = await hub.traces.submit_feedback(
        trace_ids[question], rating=-1, label="wrong-answer", comment=comment
    )
    print(f"feedback {feedback.id} on: {question[:50]}")

feedback 1f22a6ee-1ae6-4910-81d7-82040b36ec9a on: My grinder is making a loud clicking noise. Is it 
feedback 882fd83a-bbc5-417f-b1d2-2b20b83e0c7e on: Where is my order BR-1101? It should have arrived 


---

## Step 6 — Build the dataset

A **dataset** is a list of examples. Each one is an input, plus the criteria a judge grades the
answer against. Built from feedback, each thumbs-down becomes one example, and its comment
becomes that example's criteria.

Two kinds of example go in:

- **The two feedback cases.** These are what the rewrite must fix.
- **Two regression checks.** These are questions the agent already answers well. A rewrite that
  fixes one thing and breaks another loses points on them.

The dataset also gets one **overall feedback** text. The judge grades every answer against it,
and the optimizer reads it when it drafts. So it describes what a correct answer *contains*.
It names the tools only as the way to get there. The judge sees only the answer, never which
tools ran, so it cannot grade a process.

### In the dashboard

Go to **Observability → Feedback**, tick the two thumbs-down rows, and click
**Improve from feedback**. That dialog builds the dataset *and* starts the optimize run of
step 7 in one go.

| Field | What to enter |
|---|---|
| **Dataset name** | `brewline-support-misses` |
| **Overall feedback** | the `OVERALL_FEEDBACK` text from the next cell |
| **Prompt** | `brewline-support` |

To add the regression checks, open the dataset under **Evaluations → Datasets** and use
**Add example** twice, with the input and criteria from `REGRESSION_CHECKS` in the next cell.

### The same thing in code

**Setup.** It looks for the dataset by name first. Only a new dataset gets the regression
checks, so a second run adds nothing twice.

In [12]:
DATASET = "brewline-support-misses"

OVERALL_FEEDBACK = (
    "A correct answer states the Brewline policy that applies and what it means for this "
    "customer's order (dates, whether the item was used), then the exact next step the "
    "customer must take. It never gives general advice that is not in Brewline's "
    "policies. The policies come from the search_policies tool and the order from the "
    "lookup_order tool."
)

REGRESSION_CHECKS = [
    {
        "question": QUESTIONS[1],
        "criteria": "BR-1077 was delivered on 20 October, 6 days ago, so the 14-day window for a used grinder is still open. Correct answer: yes, it can be returned, but a used grinder is refunded as store credit, not money; to start, reply to the order confirmation email.",
    },
    {
        "question": QUESTIONS[2],
        "criteria": "BR-1088 is an unused kettle delivered on 2 October, 24 days ago, inside the 30-day window. Correct answer: yes, a full refund to the original payment method; to start, reply to the order confirmation email.",
    },
]

found = [d for d in await hub.datasets.list() if d.name == DATASET]
if found:
    dataset_id = found[0].id
    print(f"dataset {DATASET} already exists")
else:
    feedback_ids = [row.id for row in await thumbs_down_on(prompt.id)]
    built = await hub.datasets.build_from_feedback(DATASET, feedback_ids, overall_feedback=OVERALL_FEEDBACK)
    dataset_id = built.id
    print(f"dataset {DATASET}: {built.example_count} examples from feedback")
    await hub.datasets.add_examples(
        dataset_id,
        [{"input": {"question": c["question"], "today": TODAY}, "criteria": c["criteria"]} for c in REGRESSION_CHECKS],
    )
    print(f"+ {len(REGRESSION_CHECKS)} regression checks")

print("examples now:", len((await hub.datasets.get(dataset_id)).examples))

dataset brewline-support-misses: 2 examples from feedback
+ 2 regression checks
examples now: 4


---

## Step 7 — Let the optimizer draft rewrites

The optimize run drafts three rewrites, named `candidate-A`, `candidate-B` and `candidate-C`.
It scores each of them, and the production version, on every example in the dataset. The
strong model writes the rewrites and the same strong model grades.

### In the dashboard

If you used **Improve from feedback** in step 6, the run has already started. Otherwise open
the dataset under **Evaluations → Datasets** and click **Optimize**.

| Field | What to enter |
|---|---|
| **Prompt** | `brewline-support` |
| **Model** | `gpt-4o-mini` |
| **Judge model** | `gpt-4.1` |
| **Optimizer model** | `gpt-4.1` |
| **Number of rewrites** | 3 |

### The same thing in code

**Setup.** An optimize run costs real model calls, so this cell reuses an optimize run of this
prompt on this dataset when one already exists. Then it waits for the run to finish. It takes
about a minute.

Each optimize run drafts different rewrites, and often none of them wins in step 10. Then set
`NEW_ROUND = True` and run steps 7 to 10 again: the cell starts a new optimize run on the same
dataset instead of reusing the old one.

In [13]:
import asyncio

#: True starts a new optimize run, for a second round when step 10 finds no winner.
NEW_ROUND = False

existing = [
    r for r in (await hub.runs.list(prompt_id=prompt.id, dataset_id=dataset_id)).data
    if r.kind == "optimize" and r.status in ("queued", "running", "succeeded")
] if not NEW_ROUND else []
if existing:
    run_id = existing[0].id
    print("reusing optimize run", run_id)
else:
    started = await hub.optimize.start(
        prompt.id,
        dataset_id=dataset_id,
        models=[MODEL],
        judge_model=STRONG_MODEL,
        optimizer_model=STRONG_MODEL,
        draft_count=3,
    )
    run_id = started.run_id
    print("started optimize run", run_id)

while (run := await hub.runs.get(run_id)).status in ("queued", "running"):
    await asyncio.sleep(3)
print("run", run.status)

started optimize run 20ab7ede-9ef2-4f78-a87b-aeb8082c969a
run succeeded


**Check.** The run's leaderboard: the judge's average score (0–100) and how many of the four
examples passed.

![The run report: three candidates and v1 production, each with its average score and pass count](https://docs.acruxcore.com/img/tutorials/improve-a-rag-agent-from-feedback/optimize-leaderboard.png)

In [14]:
report = await hub.runs.get_report(run_id)
print("variant        score  passed")
for key in report.leaderboard:
    cell = next(c for c in report.cells if c["cellKey"] == key)
    passed = round((cell["passRate"] or 0) * cell["scoredCount"])
    print(f"{cell['variantLabel']:<13} {cell['avgScore'] or 0:6.1f}  {passed}/{cell['exampleCount']}")

variant        score  passed
candidate-A     57.5  1/4
candidate-B     47.5  1/4
production      42.5  0/4
candidate-C     40.0  0/4


Look at the `production` row. It fails most examples, even though the real agent answers the
two regression checks well in step 4. That is not a bug. Each example got one model call with
no tools, so the model could not look up an order or search a policy.

So this table ranks wording only. It cannot show whether a rewrite makes the agent search
before it answers. The next two steps test that.

---

## Step 8 — Commit each rewrite as a version

To run your agent on a rewrite, the rewrite must exist as a prompt version. Committing it
moves **no alias**, so `production` keeps serving the old version to your users the whole
time.

### In the dashboard

Open the run under **Evaluations → Runs**, open a candidate's cell, and click
**Test in your pipeline**, then **Commit as a version**. There is nothing to type. The dialog
shows the version number and a `compare_local` snippet with your ids filled in.

| Field | What to enter |
|---|---|
| **Candidate** | each of `candidate-A`, `candidate-B`, `candidate-C` in turn |
| **Button** | **Commit as a version** |

![The Test in your pipeline dialog: committed as brewline-support v2, with a Python compare_local snippet](https://docs.acruxcore.com/img/tutorials/improve-a-rag-agent-from-feedback/test-in-your-pipeline.png)

### The same thing in code

**Setup.** `commit_candidate` is safe to call twice. A candidate that was committed before
returns the same version, with `created` set to `False`.

In [15]:
candidates = [c for c in run.grid if c.get("promptCandidateId")]

variants: Dict[str, Optional[Dict[str, int]]] = {"production": None}  # None = no pin
for cell in candidates:
    committed = await hub.runs.commit_candidate(run_id, cell["promptCandidateId"])
    variants[cell["variantLabel"]] = {committed.prompt_name: committed.version_number}
    state = "committed" if committed.created else "already committed"
    print(f"{cell['variantLabel']} -> {committed.prompt_name} v{committed.version_number} ({state})")

candidate-A -> brewline-support v2 (committed)
candidate-B -> brewline-support v3 (committed)
candidate-C -> brewline-support v4 (committed)


---

## Step 9 — Run the whole agent on each rewrite

**Your app.** This is the cell a team would keep, for example in CI before every promote.

`compare_local` starts one local run per variant, all at the same time, over the same dataset.
For each example it calls `agent`, which calls your unchanged `ask`. Inside the `candidate-A`
run, every render of `brewline-support` returns candidate-A's version.

`hub.experiments.judge` grades each answer on the server, with the same criteria and overall
feedback the optimize run used. So the two sets of scores mean the same thing.

In [16]:
async def agent(example_input, ctx):
    answer, _ = await ask(hub, example_input["question"], trace=ctx.trace)
    return answer


comparison = await hub.experiments.compare_local(
    dataset_id,
    task=agent,
    scorers=[hub.experiments.judge(STRONG_MODEL)],
    variants=variants,
)
print(comparison)

score  production  candidate-A  candidate-B  candidate-C
judge         50%          50%          50%          50%
(pass rate per score)
compare production → candidate-A: https://acruxcore.com/evaluations/runs/56af5a55-2158-44d2-890a-6a686a3fb506/compare?target_run=8f94930c-41bb-4466-8fff-1081b2ddef29
compare production → candidate-B: https://acruxcore.com/evaluations/runs/56af5a55-2158-44d2-890a-6a686a3fb506/compare?target_run=479952c2-f52d-47ff-bddc-aa36563bce54
compare production → candidate-C: https://acruxcore.com/evaluations/runs/56af5a55-2158-44d2-890a-6a686a3fb506/compare?target_run=3fcb71d1-db02-40ba-9f4c-b007cc2661c5


Each `compare` link opens **Compare per example** in the dashboard, from production to that
candidate:

![Compare per example, production to a winning candidate: the grinder question moves from Fail to Pass, and the candidate's answer names the stone between the burrs](https://docs.acruxcore.com/img/tutorials/improve-a-rag-agent-from-feedback/compare-per-example.png)

**Check.** The same per-example view from the API: for every variant, which tools the agent
called on each question, and the judge's score. This is the part the optimizer could not see.

In [17]:
SHORT = {GRINDER: "grinder clicking", SHIPPING: "BR-1101 shipping", QUESTIONS[1]: "BR-1077 (check)", QUESTIONS[2]: "BR-1088 (check)"}

await hub.gateway.flush()
for variant, result in comparison.runs.items():
    print(f"\n{variant}  (pass rate {result.summary['judge'].pass_rate:.0%})")
    for item in result.items:
        detail = await hub.traces.get(item.trace_id)
        tools = ", ".join(tool_calls(detail.spans)) or "no tool"
        judge = next(s for s in item.scores if s.name == "judge")
        verdict = "pass" if judge.passed else "FAIL"
        print(f"  {SHORT[item.input['question']]:<18} {verdict} {round(judge.score * 100):>3}  {tools}")


production  (pass rate 50%)
  BR-1101 shipping   FAIL  40  lookup_order
  grinder clicking   FAIL  40  no tool
  BR-1077 (check)    pass  85  lookup_order, search_policies
  BR-1088 (check)    pass  80  lookup_order, search_policies

candidate-A  (pass rate 50%)
  BR-1101 shipping   FAIL  40  lookup_order
  grinder clicking   FAIL  40  no tool
  BR-1077 (check)    pass  90  lookup_order, search_policies
  BR-1088 (check)    pass 100  lookup_order, search_policies

candidate-B  (pass rate 50%)
  BR-1101 shipping   FAIL  50  lookup_order
  grinder clicking   FAIL  40  search_policies
  BR-1077 (check)    pass  85  lookup_order, search_policies
  BR-1088 (check)    pass  90  lookup_order, search_policies

candidate-C  (pass rate 50%)
  BR-1101 shipping   FAIL  60  lookup_order
  grinder clicking   FAIL  40  search_policies
  BR-1077 (check)    pass  90  search_policies, lookup_order
  BR-1088 (check)    pass  85  lookup_order, search_policies


Read it row by row, not only the pass rates. With its tools, production already passes the two
regression checks. So the question for each rewrite is narrow: does it pass a feedback case
that production fails, **without** failing a check that production passes?

Then look at the tools column. A rewrite can make the agent call `search_policies` and still
fail, because the search ran but the answer did not use what came back. That is a different
problem from a rewrite that never searches, and only this view shows which one you have.

With four examples, one answer moves a pass rate by 25 points, and a second run can give a
different answer. So read the per-example rows before you trust a rate.

---

## Step 10 — Promote the winner

Promote a candidate only if it beats production **with the tools running**. In step 9 it must
pass more examples than production, and it must still pass every example production passed.
A higher mean score with no extra pass is not enough: it usually means a failing answer got a
little less wrong.

Promoting moves the `production` alias to the version step 8 committed. Nothing new is
committed, so the wording that ships is the wording you tested.

### In the dashboard

In the run report, open the winning candidate's cell and click **Promote to production**.

| Field | What to enter |
|---|---|
| **Candidate** | the winner from step 9, e.g. `candidate-A` |
| **Alias** | `production` |

![The brewline-support versions tab after a second round: production on v6, staging still on v1, and the other candidates' versions with no alias](https://docs.acruxcore.com/img/tutorials/improve-a-rag-agent-from-feedback/prompt-versions.png)

### The same thing in code

**Setup.** It picks the winner from step 9's rows, not from the optimizer's table. When no
candidate beats production, it promotes nothing and says so. That is a normal outcome. Then
set `NEW_ROUND = True` in step 7 and run steps 7 to 10 again, or improve the feedback first.
Promoting the same candidate twice moves the alias to the same version, so a second run changes
nothing.

On a second run of the whole notebook, `production` already points at the winner, so the
baseline in step 9 is the promoted version.

In [18]:
def passes(variant: str) -> set:
    """The ids of the examples the judge passed in one variant. A notebook helper, not SDK."""
    return {
        item.example_id
        for item in comparison.runs[variant].items
        if any(s.name == "judge" and s.passed for s in item.scores)
    }


baseline = passes("production")
eligible = [
    v for v in variants
    if v != "production" and baseline <= passes(v) and len(passes(v)) > len(baseline)
]

promoted = None
if eligible:
    best = max(eligible, key=lambda v: comparison.runs[v].summary["judge"].mean or 0)
    cell = next(c for c in candidates if c["variantLabel"] == best)
    result = await hub.runs.promote_candidate(run_id, cell["promptCandidateId"], alias="production")
    promoted = result.version["versionNumber"]
    print(f"{best} passes {len(passes(best))}/4, production {len(baseline)}/4: production -> v{promoted}")
else:
    print(f"no candidate passes more than production's {len(baseline)}/4 without failing one of them")
    print("nothing promoted")

no candidate passes more than production's 2/4 without failing one of them
nothing promoted


**Check.** Ask the grinder question again, the way a real user would. `ask` still asks for
`production`, with no pin anywhere, so this is what users get now.

One thing to know first. The SDK caches each render for 60 seconds, per prompt, alias and
variables (`AcruxCore(cache_ttl=...)`, in milliseconds). When an entry is older than that, the
next call still returns it once, and fetches the new version in the background. So a running
app picks up a promote within about a minute, with no restart and no deploy.

It also means `hub` may still hold the old version right now. So this check makes a second
client with the cache turned off, `cache_ttl=0`. It sees exactly what the server serves.

![The new grinder trace: an LLM span, a search_policies tool span and a second LLM span](https://docs.acruxcore.com/img/tutorials/improve-a-rag-agent-from-feedback/trace-after-promote.png)

In [19]:
nocache = AcruxCore(cache_ttl=0)  # every render goes to the server

cached = await hub.prompts.render(PROMPT, "production", {"question": "-", "today": TODAY})
live = await nocache.prompts.render(PROMPT, "production", {"question": "-", "today": TODAY})
print(f"hub, from its cache:    v{cached.version_number}")
print(f"the server serves now:  v{live.version_number}\n")

answer, trace_id = await ask(nocache, GRINDER)
await nocache.gateway.flush()
print("A:", answer)
print("tools:", ", ".join(tool_calls((await nocache.traces.get(trace_id)).spans)) or "no tool")

hub, from its cache:    v1
the server serves now:  v1

A: A loud clicking noise from your grinder can be concerning, but it doesn't always mean it's broken. It could be caused by something as simple as a foreign object stuck in the burrs or a need for cleaning. I recommend checking for any blockages and cleaning your grinder according to the user manual. If the noise persists, it might be best to consult a technician or check if it's still under warranty.
tools: no tool


If step 10 promoted nothing, both lines say v1, and the answer is production's old answer.
That is what this notebook's saved run shows: no candidate won its first round. The screenshots
come from a second round, where a candidate did win.

If step 10 promoted a winner, the second line shows its version. Read the answer as **one
sample**. Each call is a new draw from the model, so one call can differ from what the winner
did in step 9. It may skip the search, or search and still miss the point. The trace shows
which version ran, so compare the answer with the winner's row in step 9.

That is the limit of a four-example test with one call per example. Two things make the
decision safer: run `compare_local` twice before you promote, and add more examples of the
question you are fixing. A rewrite that passes the question on every run has really fixed it.

If a question is still wrong after the promote, that is the next round: mark its new trace,
add it to the dataset, and run steps 7 to 10 again.

---

## Step 11 — Ways to get this wrong

Four real mistakes, each triggered on purpose. None of these cells is app code. Two of them
fail **silently**: no error, just the wrong version. Those are the ones worth remembering.

### A misspelled prompt name in a pin

**Broken on purpose.** These cells render through `nocache` from step 10, so no cached entry
can hide what the server returns.

A pin is keyed by the prompt's exact name. Here it says
`brewline_support`, with an underscore. Nothing renders a prompt by that name, so the pin
never applies, and your "candidate" run quietly uses production. In `compare_local`, the
candidate column would then show production's scores under the candidate's name.

In [20]:
from acruxcore import pin_prompts, pinned_version

production_v = live.version_number
other_v = next(p[PROMPT] for p in variants.values() if p and p[PROMPT] != production_v)

with pin_prompts({"brewline_support": other_v}):  # underscore - not the prompt's name
    rendered = await nocache.prompts.render(PROMPT, "production", {"question": "-", "today": TODAY})
print(f"asked for v{other_v}, got v{rendered.version_number} (production) - and no error")

with pin_prompts({PROMPT: other_v}):  # the exact name
    rendered = await nocache.prompts.render(PROMPT, "production", {"question": "-", "today": TODAY})
print(f"with the exact name: v{rendered.version_number}")

asked for v2, got v1 (production) - and no error
with the exact name: v2


The fix is to build the pin from the server's answer, as step 8 does with
`{committed.prompt_name: committed.version_number}`, instead of typing the name.

### A version that does not exist

**Broken on purpose.** This one does fail loudly. The pin reaches the server, and the server
answers that the prompt has no such version. The reason is in the error message.

In [21]:
from acruxcore import AcruxCoreError

try:
    with pin_prompts({PROMPT: 999}):
        await hub.prompts.render(PROMPT, "production", {"question": "-", "today": TODAY})
except AcruxCoreError as err:
    print(f"{err.status_code}: {err}")

404: acruxcore API error 404 for "brewline-support/production": Version 999 not found for prompt "brewline-support"


### A thread you start yourself loses the pin

**Broken on purpose.** The pin lives in a context variable. `asyncio` tasks and
`asyncio.to_thread` copy the context, so a pin reaches the code they run. A plain
`threading.Thread` starts with an empty context, so code inside it sees no pin, and renders
production. No error, again.

In [22]:
import threading

seen = {}
with pin_prompts({PROMPT: other_v}):
    seen["in the task"] = pinned_version(PROMPT)
    seen["in asyncio.to_thread"] = await asyncio.to_thread(pinned_version, PROMPT)
    thread = threading.Thread(target=lambda: seen.__setitem__("in threading.Thread", pinned_version(PROMPT)))
    thread.start()
    thread.join()

for where, version in seen.items():
    print(f"{where:<22} pinned version: {version}")

in the task            pinned version: 2
in asyncio.to_thread   pinned version: 2
in threading.Thread    pinned version: None


If your agent hands work to a thread pool, use `asyncio.to_thread`, or copy the context with
`contextvars.copy_context().run(...)` when you start the thread.

### The judge outside a local run

**Broken on purpose.** `hub.experiments.judge(...)` grades against the dataset of the run it
is part of. Called on its own, there is no run, so there is no example or criteria to grade
against. It only works as a scorer inside `run_local` or `compare_local`.

In [23]:
from acruxcore.local_experiments import LocalExample

judge = hub.experiments.judge(STRONG_MODEL)
try:
    await judge("Unplug the grinder and brush the burrs.", LocalExample(id="x", input={}, expected=None, criteria=None, trace_id="x"))
except RuntimeError as err:
    print("RuntimeError:", err)

RuntimeError: acruxcore: a judge scorer only works inside experiments.run_local.


---

## Step 12 — Close the client

**Your app.** Tool spans are reported in the background. Close each client when you finish, or
the last spans may never arrive. In a script, `async with AcruxCore() as hub:` does this for
you. A notebook has no block to leave, so call it yourself, for `hub` and for the `nocache`
client from step 10.

In [24]:
await hub.gateway.aclose()
await nocache.gateway.aclose()
print("closed - pending trace data flushed")

closed - pending trace data flushed


---

## What of this actually ships

Only the cells marked **Your app**. Put together, the agent is this:

```python
hub = AcruxCore()

milvus = MilvusClient(MILVUS_URI)          # indexed by your ingestion job, not the agent
embed = model.DefaultEmbeddingFunction()

def search_policy_store(query, k=4): ...   # embed the query, then milvus.search

@acrux.tool
async def search_policies(query: str): ...
@acrux.tool
async def lookup_order(order_id: str): ...

async def ask(hub, question, trace=None):
    rendered = await hub.prompts.render("brewline-support", "production", {"question": question, "today": today})
    result = await hub.gateway.run_prompt_with_tools(
        rendered, tools=[search_policies, lookup_order], trace={"name": "brewline-support", **(trace or {})}
    )
    return result.content, result.trace_id
```

And the test a team keeps, in CI or beside a release checklist, is step 9:

```python
comparison = await hub.experiments.compare_local(
    dataset_id,
    task=agent,
    scorers=[hub.experiments.judge("gpt-4.1")],
    variants={"production": None, "candidate-A": {"brewline-support": 2}},
)
```

Everything else was scaffolding:

- **Setup** cells created the prompt, the feedback, the dataset, the optimize run and the
  committed versions. The dashboard does each of those jobs, and you do them once, not on
  every request.
- **Check** cells read traces and reports back so you could see what happened.
- `thumbs_down_on`, `tool_calls` and `passes` are notebook helpers, not SDK functions.
- The `nocache` client exists only so the checks can see past the render cache. Your app keeps
  the default cache.
- The indexing cell belongs in a separate ingestion job that runs when a policy changes.
- The inline `POLICIES` and `ORDERS` stand in for your own documents and order database.

The one habit worth keeping: for an agent, the optimizer's table drafts the candidates, and
`compare_local` with your real tools decides which one ships.